In [ ]:
import polars as pl
from pathlib import Path

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups = [
    "plants_plt_solanaceae"
]

In [ ]:
for g in groups:
    gdir = BASE / g

    tsv = pl.read_csv(gdir / f"{g}.tsv", separator="\t")
    atlas = pl.read_csv(gdir / f"{g}_summary_atlas.tsv", separator="\t")

    atlas = atlas.with_columns(
        pl.col("UP").str.replace("PLT-", "").alias("UP"),
        pl.col("DOWN").str.replace("PLT-", "").alias("DOWN"),
    )

    atlas = atlas.with_columns(
        pl.concat_str([pl.col("UP"), pl.col("DOWN"), pl.col("Polarity")], separator="|").alias("gene_pair")
    )

    atlas.write_csv(gdir / f"{g}_summary_atlas2.tsv", separator="\t")

    gene_pairs = atlas.group_by("gene_pair").agg(
        pl.len().alias("igr_count"),
        pl.col("Length").mean().alias("mean_len"),
        pl.col("Length").median().alias("median_len"),
        pl.col("Length").std().alias("std_len"),
        pl.col("Length").quantile(0.25).alias("q1_len"),
        pl.col("Length").quantile(0.5).alias("q2_len"),
        pl.col("Length").quantile(0.75).alias("q3_len"),
        pl.col("Length").quantile(0.99).alias("q99_len"),
        pl.col("Length").min().alias("min_len"),
        pl.col("Length").max().alias("max_len"),
    ).sort("igr_count", descending=True)

    for col in gene_pairs.columns:
        # if col any type is float, round to 2 decimal places
        if gene_pairs[col].dtype in [pl.Float32, pl.Float64]:
            gene_pairs = gene_pairs.with_columns(pl.col(col).round(2))

    gene_pairs.write_csv(gdir / f"gene_pairs.tsv", separator="\t")


In [ ]:
for g in groups:
    group_dir = BASE / g
    
    gene_pairs = pl.read_csv(group_dir / f"gene_pairs.tsv", separator="\t")

    gene_pairs = gene_pairs.with_columns(
        pl.col("gene_pair")
        .str.replace(r"\|[+-]{2}$", "")
        .alias("gene_pair_cleaned")
    )

    gene_pairs = gene_pairs.filter(
    pl.col("gene_pair_cleaned").count().over("gene_pair_cleaned") > 1
    ).sort("gene_pair")
    gene_pairs.write_csv(group_dir / f"gene_pairs_flipped.tsv", separator="\t")

In [ ]:
for g in groups:
    group_dir = BASE / g
    
    gene_pairs = pl.read_csv(group_dir / f"gene_pairs.tsv", separator="\t").filter(pl.col("igr_count") > 10)

    gene_pairs.sort("std_len", descending=False).write_csv(group_dir / f"gene_pairs_filtered.tsv", separator="\t")